In [5]:
import json
from rouge_score import rouge_scorer
from tqdm import tqdm
from collections import defaultdict

In [6]:
# Path to data JSON 
file_path = "fact_check_sample.json"

# Initialize ROUGE scorer
scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)

summary_scores = []  
model_scores = defaultdict(list)

In [ ]:
with open(file_path, "r") as f:
    for line in tqdm(f, desc="Processing data"):
        data = json.loads(line.strip())
        reference = data["reference"]
        sentences = " ".join(data["sentences"]) # concatenate sentences into summary
        model = data["model"]

        scores = scorer.score(reference, sentences)

        # summary-level scores
        summary_scores.append({
            "doc_id": data["doc_id"],
            "model": model,
            "rouge1": scores["rouge1"].fmeasure,
            "rouge2": scores["rouge2"].fmeasure,
            "rougeL": scores["rougeL"].fmeasure
        })

        # model-level scores
        model_scores[model].append({
            "rouge1": scores["rouge1"].fmeasure,
            "rouge2": scores["rouge2"].fmeasure,
            "rougeL": scores["rougeL"].fmeasure
        })

In [8]:
# Calculate model-level avg scores
model_avg_scores = {}
for model, scores in model_scores.items():
    rouge1_avg = sum(score["rouge1"] for score in scores) / len(scores)
    rouge2_avg = sum(score["rouge2"] for score in scores) / len(scores)
    rougeL_avg = sum(score["rougeL"] for score in scores) / len(scores)
    model_avg_scores[model] = {
        "rouge1_avg": rouge1_avg,
        "rouge2_avg": rouge2_avg,
        "rougeL_avg": rougeL_avg
    }

In [ ]:
print("Individual Summary Scores:")
for score in summary_scores:
    print(f"Doc ID: {score['doc_id']}, Model: {score['model']}, ROUGE-1: {score['rouge1']:.4f}, ROUGE-2: {score['rouge2']:.4f}, ROUGE-L: {score['rougeL']:.4f}")

In [ ]:
print("\nModel-Level Average Scores:")
for model, avg_score in model_avg_scores.items():
    print(f"Model: {model}, ROUGE-1: {avg_score['rouge1_avg']:.4f}, ROUGE-2: {avg_score['rouge2_avg']:.4f}, ROUGE-L: {avg_score['rougeL_avg']:.4f}")

In [13]:
with open("factcheck_rouge_scores.json", 'w') as f:
    json.dump(summary_scores, f, indent=4)
    json.dump(model_avg_scores, f, indent=4)